## Medical Reasoning LLM - Data Cleaning

### Objective

Prepare the medical reasoning dataset for downstream SFT and RLVR.

### Dataset

- Source: https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT
- Key fields:
    - Question
    - Complex_CoT
    - Response

### Pipeline

- Fetch Raw Dataset
- Quality Control
- Deduplication
- Format Validation
- Train / Validation / Test Split
- Clean Dataset

## Fetch Raw Dataset

In [28]:
!git -C /content/ml-ai-portfolio pull origin main

remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 10 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.29 KiB | 1.14 MiB/s, done.
From https://github.com/xueqingnie/ml-ai-portfolio
 * branch            main       -> FETCH_HEAD
   b47d065..2e21db0  main       -> origin/main
Updating b47d065..2e21db0
Fast-forward
 .../notebooks/01_data_cleaning.ipynb               | 162 +++++++++++----------
 01-medical-reasoning-llm/src/data.py               |   4 +
 2 files changed, 91 insertions(+), 75 deletions(-)


In [29]:
!cat /content/ml-ai-portfolio/01-medical-reasoning-llm/src/data.py

from datasets import load_dataset

def fetch_hf_dataset(
    dataset_name: str,
    config: str = None,
    split: str = "train",
):
    """
    Load a dataset from Hugging Face Hub.
    
    Parameters:
    dataset_name : str
        Name of the dataset on Hugging Face.
    config : str, optional
        Dataset configuration, such as "en", "zh", "en_mix", or "zh_mix".
    split : str, default="train"
        Dataset split to load, such as "train", "test", or "validation".
    """
            
    return load_dataset(
        dataset_name,
        config,
        split=split,
    )


In [30]:
import importlib
import src.data

importlib.reload(src.data)

from src.data import fetch_hf_dataset

import inspect
print(inspect.signature(fetch_hf_dataset))

(dataset_name: str, config: str = None, split: str = 'train')


In [53]:
dataset = fetch_hf_dataset(
    "FreedomIntelligence/medical-o1-reasoning-SFT",
    config="en",
    split="train",
)

In [54]:
df = dataset.to_pandas()

In [55]:
df.head()

,Question,Complex_CoT,Response
0,Given the symptoms of sudden weakness in the l...,"Okay, let's see what's going on here. We've go...",The specific cardiac abnormality most likely t...
1,A 33-year-old woman is brought to the emergenc...,"Okay, let's figure out what's going on here. A...","In this scenario, the most likely anatomical s..."
2,A 61-year-old woman with a long history of inv...,"Okay, let's think about this step by step. The...",Cystometry in this case of stress urinary inco...
3,A 45-year-old man with a history of alcohol us...,"Alright, let’s break this down. We have a 45-y...",Considering the clinical presentation of sudde...
4,A 45-year-old man presents with symptoms inclu...,"Okay, so here's a 45-year-old guy who's experi...",Based on the clinical findings presented—wide-...


## Data Exploration

In [57]:
print("Dataset shape:", df.shape)
print(df.columns.tolist())

Dataset shape: (19704, 3)
['Question', 'Complex_CoT', 'Response']


In [56]:
df.describe()

,Question,Complex_CoT,Response
count,19704,19704,19704
unique,19679,19704,19703
top,What is the name of the classification propose...,"Alright, let's think this through. The symptom...",D. The first statement is false and the second...
freq,3,1,2


In [58]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19704 entries, 0 to 19703
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Question     19704 non-null  object
 1   Complex_CoT  19704 non-null  object
 2   Response     19704 non-null  object
dtypes: object(3)
memory usage: 461.9+ KB


In [67]:
# Check for missing values

print(df.isnull().sum())

Question       0
Complex_CoT    0
Response       0
dtype: int64


In [60]:
# Check for empty strings

for col in ["Question", "Complex_CoT", "Response"]:
    empty_count = df[col].astype(str).str.strip().eq("").sum()
    print(f"{col}: {empty_count} empty strings")

Question: 0 empty strings
Complex_CoT: 0 empty strings
Response: 0 empty strings


In [66]:
# Check for duplicated rows and fields

print("Duplicated rows:", df.duplicated().sum())

for col in ["Question", "Complex_CoT", "Response"]:
    duplicated_count = df[col].duplicated().sum()
    print(f"{col}: {duplicated_count} duplicated values")

Duplicated rows: 0
Question: 25 duplicated values
Complex_CoT: 0 duplicated values
Response: 1 duplicated values


In [68]:
# Check text length

df[["Question", "Complex_CoT", "Response"]].apply(
    lambda x: x.str.len()
).describe()

,Question,Complex_CoT,Response
count,19704.000000,19704.000000,19704.000000
mean,297.826786,1906.967063,638.603634
std,211.518158,408.178224,325.733800
min,51.000000,834.000000,4.000000
25%,163.000000,1621.000000,436.000000
50%,244.000000,1842.000000,570.000000
75%,358.000000,2129.000000,751.250000
max,2380.000000,5535.000000,3999.000000


In [69]:
df.nsmallest(10, "question_length")[
    ["Question", "Complex_CoT", "Response"]
]

KeyError: 'question_length'

In [ ]:
df.nsmallest(10, "cot_length")[
    ["Question", "Complex_CoT", "Response"]
]

In [ ]:
df.nlargest(5, "cot_length")[
    ["Question", "Complex_CoT", "Response"]
]